# Simple Colab Training Runner

This notebook does only what you asked:
- install required packages
- mount Google Drive
- upload and unzip `project.zip`
- run a Python training script directly (no bash wrapper) with explicit data path args
- copy only newly created run folders to `/content/drive/MyDrive/PSBD/runs` after each run

Drive paths used:
- root: `/content/drive/MyDrive/PSBD`
- preprocessed data: `/content/drive/MyDrive/PSBD/preprocessed_data`
- raw data: `/content/drive/MyDrive/PSBD/raw_data`
- runs: `/content/drive/MyDrive/PSBD/runs`

In [ ]:
# Install packages not guaranteed to exist in Colab
!pip -q install lightning

In [ ]:
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive", force_remount=True)

In [ ]:
ROOT_DRIVE_DIR = Path("/content/drive/MyDrive/PSBD")
DRIVE_PREPROCESSED_DIR = ROOT_DRIVE_DIR / "preprocessed_data"
DRIVE_RAW_DIR = ROOT_DRIVE_DIR / "raw_data"
DRIVE_RUNS_DIR = ROOT_DRIVE_DIR / "runs"

for path in [ROOT_DRIVE_DIR, DRIVE_PREPROCESSED_DIR, DRIVE_RAW_DIR, DRIVE_RUNS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def upload_and_unzip_project(
    zip_name: str = "project.zip", extract_root: Path = Path(".")
) -> Path:
    zip_path = extract_root / zip_name
    if not zip_path.exists():
        uploaded = files.upload()
        if zip_name not in uploaded:
            raise FileNotFoundError(
                f"Upload a file named {zip_name}. Uploaded: {list(uploaded.keys())}"
            )
        with open(zip_path, "wb") as f:
            f.write(uploaded[zip_name])

    subprocess.run(
        ["unzip", "-q", "-o", str(zip_path), "-d", str(extract_root)], check=True
    )

    # Always use current folder as project root.
    return Path(".").resolve()


def copy_new_runs_to_drive(
    project_dir: Path, before_names: set[str], drive_runs_dir: Path = DRIVE_RUNS_DIR
):
    local_runs_dir = project_dir / "runs"
    local_runs_dir.mkdir(parents=True, exist_ok=True)
    after_names = {p.name for p in local_runs_dir.iterdir() if p.is_dir()}
    new_names = sorted(after_names - before_names)

    if not new_names:
        print("No new run directory detected.")
        return []

    copied = []
    for name in new_names:
        src = local_runs_dir / name
        dst = drive_runs_dir / name
        shutil.copytree(src, dst, dirs_exist_ok=True)
        copied.append(str(dst))

    print("Copied runs to Drive:")
    for p in copied:
        print(" -", p)
    return copied


def run_training_script(
    project_dir: Path,
    script_name: str,
    script_args: list[str] | None = None,
    data_root: Path = DRIVE_PREPROCESSED_DIR,
):
    script_path = project_dir / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")

    local_runs_dir = project_dir / "runs"
    local_runs_dir.mkdir(parents=True, exist_ok=True)
    before_names = {p.name for p in local_runs_dir.iterdir() if p.is_dir()}

    cmd = ["python", str(script_path)]
    cmd += ["--data-root", str(data_root)]
    if script_args:
        cmd += script_args

    print("Running command:")
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(project_dir), check=True)

    return copy_new_runs_to_drive(project_dir, before_names, DRIVE_RUNS_DIR)

In [ ]:
# Setup project folder from uploaded zip
PROJECT_DIR = upload_and_unzip_project(zip_name="project.zip", extract_root=Path("."))
print("PROJECT_DIR =", PROJECT_DIR)

# Optional check
print("Drive preprocessed_data dir:", DRIVE_PREPROCESSED_DIR)
print("Drive raw_data dir:", DRIVE_RAW_DIR)
print("Drive runs dir:", DRIVE_RUNS_DIR)

In [ ]:
# Smoke loop: run all scripts across all datasets with max_epochs=5
datasets = ["cifar10", "cifar100", "gtsrb"]

for dataset_name in datasets:
    print(f"\n=== SMOKE RUNS: {dataset_name} ===")

    # Benign
    run_training_script(
        project_dir=PROJECT_DIR,
        script_name="benign_train_resnet.py",
        script_args=[
            "--dataset-name",
            dataset_name,
            "--max-epochs",
            "5",
            "--early-stopping-patience",
            "5",
        ],
    )

    # BadNet + PSBD
    run_training_script(
        project_dir=PROJECT_DIR,
        script_name="badnet_train_resnet.py",
        script_args=[
            "--dataset-name",
            dataset_name,
            "--max-epochs",
            "5",
            "--early-stopping-patience",
            "5",
            "--run-psbd",
            "--psbd-dropout-rates",
            "0.8",
            "--psbd-target-fprs",
            "0.25",
            "--psbd-selection-fpr",
            "0.25",
        ],
    )

    # WaNet + PSBD
    run_training_script(
        project_dir=PROJECT_DIR,
        script_name="wanet_train_resnet.py",
        script_args=[
            "--dataset-name",
            dataset_name,
            "--max-epochs",
            "5",
            "--early-stopping-patience",
            "5",
            "--run-psbd",
            "--psbd-dropout-rates",
            "0.8",
            "--psbd-target-fprs",
            "0.25",
            "--psbd-selection-fpr",
            "0.25",
        ],
    )

print("\nAll smoke runs finished.")

In [ ]:
# Full-training loop: run all scripts across all datasets with default epochs
datasets = ["cifar10", "cifar100", "gtsrb"]

for dataset_name in datasets:
    print(f"\n=== FULL TRAINING: {dataset_name} ===")

    # Benign
    run_training_script(
        project_dir=PROJECT_DIR,
        script_name="benign_train_resnet.py",
        script_args=[
            "--dataset-name",
            dataset_name,
            "--early-stopping-patience",
            "5",
        ],
    )

    # BadNet + PSBD
    run_training_script(
        project_dir=PROJECT_DIR,
        script_name="badnet_train_resnet.py",
        script_args=[
            "--dataset-name",
            dataset_name,
            "--early-stopping-patience",
            "5",
            "--run-psbd",
            "--psbd-dropout-rates",
            "0.8",
            "--psbd-target-fprs",
            "0.25",
            "--psbd-selection-fpr",
            "0.25",
        ],
    )

    # WaNet + PSBD
    run_training_script(
        project_dir=PROJECT_DIR,
        script_name="wanet_train_resnet.py",
        script_args=[
            "--dataset-name",
            dataset_name,
            "--early-stopping-patience",
            "5",
            "--run-psbd",
            "--psbd-dropout-rates",
            "0.8",
            "--psbd-target-fprs",
            "0.25",
            "--psbd-selection-fpr",
            "0.25",
        ],
    )

print("\nAll full trainings finished.")